In [ ]:
# Cell 1 — Setup
"""
06_ablations.ipynb
==================
Interactive ablation analysis.

Canonical CLI path for the paper-scale study: `python scripts/experiments/run_ablation.py ...`
This notebook remains useful for single-seed exploratory runs and visualization.
The maintained ablation set now also includes the `split_late_encoder` backbone variant.
"""

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"
TABLES_DIR = PROJECT_ROOT / "tables"
NOTEBOOK_FIG_DIR = FIGURES_DIR / "notebooks"
NOTEBOOK_RESULTS_DIR = RESULTS_DIR / "notebooks"
NOTEBOOK_FIG_DIR.mkdir(parents=True, exist_ok=True)
NOTEBOOK_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from tgnn_solv.config import TGNNSolvConfig
from tgnn_solv.data import make_loaders, PROCESSED_DIR
from tgnn_solv.ablation import run_ablation_study

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
print(f"Project root: {PROJECT_ROOT}")
print(f"Device: {DEVICE}")


## How to read ablations quantitatively

An ablation measures the contribution of an architectural decision via the change in a metric
relative to the full model. For any metric \(m\) and variant \(v\), it is useful to track both
the absolute and relative shift:

$$
\Delta m_v = m_v - m_{\mathrm{full}},
$$

$$
\Delta m_v^{\mathrm{rel}} =
\frac{m_v - m_{\mathrm{full}}}{m_{\mathrm{full}}} \cdot 100\%.
$$

For error metrics such as MAE or RMSE, the sign is interpreted as

$$
\Delta \mathrm{MAE}_v > 0 \Rightarrow \text{the variant is worse than the full model},
$$

$$
\Delta \mathrm{MAE}_v < 0 \Rightarrow \text{the variant is better than the full model}.
$$

If multiple seeds are available, the natural aggregates are the mean and the between-seed variability:

$$
\bar m_v = \frac{1}{S} \sum_{s=1}^{S} m_{v,s},
$$

$$
s_v = \sqrt{\frac{1}{S-1} \sum_{s=1}^{S} (m_{v,s} - \bar m_v)^2}.
$$

The waterfall and bar chart in this notebook are best read as visualizations of \(\Delta m_v\),
not as a collection of unrelated absolute numbers.


## Step 1. Fix the reference experiment

Before launching ablations, it is important to fix one shared dataset setup and one base configuration.
Otherwise the meaning of the `\Delta` metrics becomes blurry. The variants below should be read as
controlled perturbations around one reference setup.


In [ ]:
# Cell 2 — Load data
train_df = pd.read_csv(PROCESSED_DIR / "train.csv")
val_df = pd.read_csv(PROCESSED_DIR / "val.csv")
test_df = pd.read_csv(PROCESSED_DIR / "test.csv")

cfg = TGNNSolvConfig(
    hidden_dim=256,
    n_gnn_layers=6,
    encoder_role_mode="shared_residual",
    encoder_role_specific_layers=2,
    n_cross_attn_layers=3,
    n_attn_heads=8,
    pair_dim=512,
    nrtl_tau_mode="ref_invT",
    use_pair_temperature_batching=True,
    pair_temperature_min_group_size=2,
    pair_temperature_group_chunk_size=4,
    epochs_phase1=1,
    epochs_phase2=2,
    epochs_phase3=1,
    warmup_epochs=1,
)

train_loader, val_loader, test_loader = make_loaders(
    train_df,
    val_df,
    test_df,
    batch_size=cfg.batch_size,
    use_pair_temperature_batching=cfg.use_pair_temperature_batching,
    pair_temperature_min_group_size=cfg.pair_temperature_min_group_size,
    pair_temperature_group_chunk_size=cfg.pair_temperature_group_chunk_size,
)


## Step 2. A single-seed preview before a large run

This notebook intentionally performs a single-seed preview. It is not the final statistics;
it is a fast way to see which variants are worth carrying into a multi-seed runner and which already
look weak at an early stage.


In [ ]:
# Cell 3 — Run ablation study (single seed — notebook preview)
# For the full multi-seed paper run, use scripts/experiments/run_ablation.py.

results = run_ablation_study(
    train_loader, val_loader, test_loader,
    device=DEVICE,
    base_cfg=cfg,
    test_df=test_df,
    seeds=[42],
    skip=[],   # Use skip=["large_512"] if memory-constrained.
)

results.to_csv(NOTEBOOK_RESULTS_DIR / "ablation_results.csv", index=False)
results


## Step 3. Identify where quality is lost

The ablations are best read not as a list of models, but as a map of architectural vulnerabilities.
If removing one block sharply increases MAE, that block is carrying real signal rather than merely
making the network cosmetically more complex.


In [ ]:
# Cell 4 — Ablation bar chart

results_sorted = results.sort_values("mae")
full_mae = results[results["ablation"] == "full"]["mae"].values[0]

fig, ax = plt.subplots(figsize=(10, 6))

names = results_sorted["name"].values
maes = results_sorted["mae"].values
deltas = maes - full_mae

colors = []
for name, delta in zip(names, deltas):
    if "full" in name:
        colors.append("steelblue")
    elif delta > 0.05:
        colors.append("tomato")
    elif delta > 0.01:
        colors.append("sandybrown")
    elif delta > -0.01:
        colors.append("lightgray")
    else:
        colors.append("mediumseagreen")

bars = ax.barh(names, maes, color=colors, edgecolor="black", linewidth=0.5)

# Reference line
ax.axvline(full_mae, color="steelblue", ls="--", lw=1.5,
           alpha=0.7, label=f"Full model ({full_mae:.3f})")

# Annotations
for bar, mae_val, delta in zip(bars, maes, deltas):
    label = f"{mae_val:.3f}"
    if abs(delta) > 0.005 and "full" not in str(bar):
        label += f" ({delta:+.3f})"
    ax.text(mae_val + 0.01, bar.get_y() + bar.get_height() / 2,
            label, va="center", fontsize=9)

ax.set_xlabel("MAE (ln x₂)", fontsize=12)
ax.set_title("Ablation Study: Component Contributions", fontsize=13)
ax.legend(loc="lower right")
ax.invert_yaxis()

plt.tight_layout()
plt.savefig(NOTEBOOK_FIG_DIR / "ablation_chart.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# Cell 5 — Delta waterfall chart

ablation_order = [
    "split_late_encoder",
    "no_nrtl",
    "no_cross_attn",
    "no_aux_losses",
    "no_curriculum",
    "no_correction",
    "no_implicit_diff",
]

fig, ax = plt.subplots(figsize=(10, 5))

names_ordered = []
deltas_ordered = []
for abl in ablation_order:
    row = results[results["ablation"] == abl]
    if len(row) == 0:
        continue
    delta = row["mae"].values[0] - full_mae
    clean_name = abl.replace("no_", "− ").replace("_", " ").title()
    names_ordered.append(clean_name)
    deltas_ordered.append(delta)

x = np.arange(len(names_ordered))
colors = ["tomato" if d > 0 else "mediumseagreen" for d in deltas_ordered]
ax.bar(x, deltas_ordered, color=colors, edgecolor="black", width=0.6)

ax.set_xticks(x)
ax.set_xticklabels(names_ordered, rotation=30, ha="right", fontsize=10)
ax.set_ylabel("ΔMAE vs full model", fontsize=11)
ax.set_title("Impact of removing each component", fontsize=13)
ax.axhline(0, color="black", lw=0.8)

for i, (d, name) in enumerate(zip(deltas_ordered, names_ordered)):
    ax.text(i, d + 0.005 * np.sign(d), f"{d:+.3f}",
            ha="center", va="bottom" if d > 0 else "top", fontsize=10)

plt.tight_layout()
plt.savefig(NOTEBOOK_FIG_DIR / "ablation_waterfall.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# Cell 6 — Scaling analysis

scaling_rows = results[results["ablation"].isin(
    ["small_128", "full", "large_512"]
)]

if len(scaling_rows) >= 2:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # MAE vs params
    ax = axes[0]
    ax.plot(scaling_rows["n_params"], scaling_rows["mae"],
            "o-", ms=10, color="steelblue", lw=2)
    for _, row in scaling_rows.iterrows():
        ax.annotate(row["name"], (row["n_params"], row["mae"]),
                    textcoords="offset points", xytext=(10, 5), fontsize=9)
    ax.set_xlabel("Parameters")
    ax.set_ylabel("MAE (ln x₂)")
    ax.set_title("Scaling: accuracy vs model size")
    ax.set_xscale("log")

    # MAE vs training time
    ax = axes[1]
    ax.plot(scaling_rows["train_time_s"] / 60, scaling_rows["mae"],
            "s-", ms=10, color="coral", lw=2)
    for _, row in scaling_rows.iterrows():
        ax.annotate(row["name"], (row["train_time_s"] / 60, row["mae"]),
                    textcoords="offset points", xytext=(10, 5), fontsize=9)
    ax.set_xlabel("Training time (minutes)")
    ax.set_ylabel("MAE (ln x₂)")
    ax.set_title("Scaling: accuracy vs compute")

    plt.tight_layout()
    plt.savefig(NOTEBOOK_FIG_DIR / "scaling_analysis.png", dpi=150)
    plt.show()


## Step 4. Turn an exploratory result into a paper-ready artifact

The final tables and summaries are useful because they convert one-off notebook findings into a form
that fits reproducible reporting. This is a convenient bridge between exploratory work and what later
ends up in an article, appendix, or internal comparison memo.


In [ ]:
# Cell 7 — LaTeX table for paper

print("Table for paper:")
print("=" * 60)

cols = ["name", "n_params", "mae", "rmse", "r2"]
table_df = results[cols].copy()
table_df["n_params"] = table_df["n_params"].apply(lambda x: f"{x / 1e6:.1f}M")
table_df = table_df.rename(columns={
    "name": "Model",
    "n_params": "Params",
    "mae": "MAE",
    "rmse": "RMSE",
    "r2": "$R^2$",
})

# Sort by MAE
table_df = table_df.sort_values("MAE")

print(table_df.to_latex(
    index=False,
    float_format="%.3f",
    caption="Ablation study results on BigSolDBv2.1 test set.",
    label="tab:ablation",
    escape=False,
))

In [ ]:
# Cell 8 — Key findings summary

print("\n" + "=" * 60)
print("  ABLATION STUDY — KEY FINDINGS")
print("=" * 60)

# Rank components by impact
impacts = []
for abl in results["ablation"].unique():
    if abl == "full":
        continue
    delta = results[results["ablation"] == abl]["mae"].values[0] - full_mae
    impacts.append((abl, delta))

impacts.sort(key=lambda x: x[1], reverse=True)

print("\nComponent importance (ranked by MAE impact when removed):\n")
for rank, (name, delta) in enumerate(impacts, 1):
    direction = "↑" if delta > 0 else "↓"
    significance = ""
    if abs(delta) > 0.05:
        significance = " *** (critical)"
    elif abs(delta) > 0.02:
        significance = " ** (important)"
    elif abs(delta) > 0.01:
        significance = " * (helpful)"
    else:
        significance = " (negligible)"

    print(f"  {rank}. {name:20s}  ΔMAE = {delta:+.3f} {direction}{significance}")

print(f"""
Interpretation guide:
  - Positive ΔMAE = removing the component makes the model WORSE
    → the component is useful
  - Negative ΔMAE = removing the component makes the model BETTER
    → the component may be harmful or unnecessary
  - |ΔMAE| < 0.01 = statistically insignificant
    → run with multiple seeds to confirm
""")